# 05: Custom Dataset Training, Quantization & Inference
Interactive notebook to prepare a custom dataset, train **tinyYOLO-det**, apply INT8 Quantization (PTQ & QAT), plot training curves, and validate predictions on custom test images.

Designed to run in local Jupyter Lab, Google Colab, or Kaggle.

In [ ]:
# %% Setup & Environment Installation
import os, sys, socket
try:
    socket.create_connection(("github.com", 80), timeout=5)
except:
    print("❌ ERROR: No Internet connection. If on Kaggle, toggle 'Internet' to ON in the sidebar settings.")
    sys.exit(1)

# Clone repository if running in a cloud VM (Colab / Kaggle)
if not os.path.exists('scripts/train.py'):
    print("🚀 Cloud Environment Detected. Cloning repository...")
    !rm -rf tinyYOLO && git clone https://github.com/ShMazumder/tinyYOLO.git
    %cd tinyYOLO
else:
    print("✓ Local Workspace Detected. Running in project root.")

# Install dependencies in editable mode
!pip install -e . -q
!pip install tqdm timm ultralytics matplotlib pyyaml opencv-python onnx onnxscript onnxruntime -q

## Option A: Local Custom Dataset Setup
Choose this option if you have raw images and labels saved locally. Run these cells to initialize directories and create your config YAML file.

In [ ]:
# %% Create directory structure (Option A)
from pathlib import Path

DATASET_NAME = "custom_data"  # Rename to your preferred folder name
dataset_dir = Path("datasets") / DATASET_NAME

folders = [
    dataset_dir / "images" / "train",
    dataset_dir / "images" / "val",
    dataset_dir / "labels" / "train",
    dataset_dir / "labels" / "val"
]

for f in folders:
    f.mkdir(parents=True, exist_ok=True)
    print(f"  ✓ Created: {f}")

data_yaml_arg = str(Path("datasets") / f"{DATASET_NAME}.yaml")
print(f"\n👉 Directory setup complete! Place your local files inside 'datasets/{DATASET_NAME}/'")

In [ ]:
# %% Write datasets/custom_data.yaml (Option A)
import yaml

# Modify these fields for your specific custom object detection task
custom_config = {
    "path": DATASET_NAME,
    "train": "images/train",
    "val": "images/val",
    "nc": 2,               # Number of classes
    "names": {
        0: "object_a",      # Rename to your actual classes
        1: "object_b"
    }
}

with open(data_yaml_arg, 'w') as f:
    yaml.dump(custom_config, f, default_flow_style=False)

print(f"✓ Local config created at: {data_yaml_arg}")

## Option B: Automated Roboflow Dataset Download
Choose this option if your dataset is hosted on Roboflow. Run these cells to download, extract, and auto-parse the class labels.

In [ ]:
# %% Download from Roboflow (Option B)
!pip install roboflow
from roboflow import Roboflow

# Define your Roboflow Private API Key here (or retrieve it from environment)
RB_KEY = "YOUR_ROBOFLOW_API_KEY_HERE"

rf = Roboflow(api_key=RB_KEY)
project = rf.workspace("shmazumder").project("crime-detection-oodyj-4mgzl")
version = project.version(1)
dataset = version.download("yolo26")

In [ ]:
# %% Parse Roboflow dataset information (Option B)
import yaml
from pathlib import Path

dataset_dir = Path(dataset.location)
DATASET_NAME = dataset_dir.name  # Typically 'crime-detection-1'
data_yaml_arg = str(dataset_dir / "data.yaml")

print(f"✓ Dataset Download Path: {dataset_dir}")
print(f"✓ DATASET_NAME resolved to: {DATASET_NAME}")
print(f"✓ data_yaml_arg resolved to: {data_yaml_arg}")

# Load class names directly from Roboflow configuration file
if Path(data_yaml_arg).exists():
    with open(data_yaml_arg) as f:
        roboflow_config = yaml.safe_load(f)
    
    nc = roboflow_config.get("nc", 0)
    names = roboflow_config.get("names", {})
    print(f"\n✓ Loaded config from Roboflow 'data.yaml':")
    print(f"  - Classes count (nc): {nc}")
    print(f"  - Class mapping (names): {names}")
else:
    print(f"⚠️ Warning: data.yaml not found at {data_yaml_arg}.")

## 1. Advanced Performance features (Automatic Caching & Multi-GPU DDP)

### A. Automatic Image Caching (RAM Caching)
Your training dataset loaders automatically compute memory requirements and benchmark them against your active system memory (`avail_gb * 0.4`). 
* **RAM Caching ON**: If the dataset fits safely in memory, it loads all preprocessed images into RAM. This eliminates CPU worker disk IO bottlenecks entirely, accelerating training speeds up to **2-3x**.
* **RAM Caching OFF**: If the dataset is too large, it dynamically streams images from disk, protecting your VM from Out-of-Memory (OOM) kernel panics. You do not need to manually configure this.

### B. Multi-GPU Distributed Training (DDP)
The training script natively supports PyTorch `DistributedDataParallel` (DDP) for high-performance multi-GPU systems. If you have multiple GPUs (e.g. 2 or 4 T4/A100 instances), **do not** run the training command via python directly. Instead, run this shell command:

```bash
torchrun --nproc_per_node=YOUR_GPU_COUNT scripts/train.py \
    --task det \
    --variant standard \
    --data crime-detection-1/data.yaml \
    --epochs 50 \
    --batch 64 \
    --pretrained
```
*(Note: Adjust `--nproc_per_node` to match your exact physical GPU count).* 

## 2. Train on Custom Dataset (Transfer Learning)
Run the training loop. We point the training script directly to your configured `data_yaml_arg` (set automatically in Option A or Option B), bypassing any manual setup. We pass `--pretrained` to use our ImageNet GhostNet weights.

> [!IMPORTANT]
> If you plan to quantize your custom model to INT8, set `VARIANT = "quantized"` below. The **quantized** variant replaces standard layers with INT8-compatible operators (like **ReLU6** and **ECA**) which prevents accuracy drop when compressing to INT8!

In [ ]:
# %% Train tinyYOLO-det standard/quantized variant
VARIANT = "quantized"  # standard | quantized (use quantized if you want to compress to INT8)
IMGSZ = 416            # Input resolution matching dataset image size
EPOCHS = 50            # Epoch count
BATCH = 32             # Batch size (set to -1 to auto-detect optimal size)
EXP_NAME = "crime-detection-yolo-run"

# Points automatically to either Local or Roboflow configuration depending on which option you ran above!
data_yaml = data_yaml_arg

!python3 scripts/train.py \
    --task det \
    --variant {VARIANT} \
    --data {data_yaml} \
    --imgsz {IMGSZ} \
    --epochs {EPOCHS} \
    --batch {BATCH} \
    --pretrained \
    --compile \
    --name {EXP_NAME}

## 3. Visualize Metrics & Loss Curves
Load the training `history.json` and plot Box Loss, Cls Loss, and validation mAP curves.

In [ ]:
# %% Plot training history
import json
from pathlib import Path
import matplotlib.pyplot as plt

runs_dir = Path("experiments/results")
run_folders = sorted(runs_dir.glob(f"*{EXP_NAME}*"))

if run_folders:
    latest_run = run_folders[-1]
    history_file = latest_run / "history.json"
    
    if history_file.exists():
        with open(history_file) as f:
            history = json.load(f)
            
        epochs = [x['epoch'] for x in history]
        box_loss = [x['box'] for x in history]
        cls_loss = [x['cls'] for x in history]
        total_loss = [x['total'] for x in history]
        map50 = [x.get('mAP50', 0) for x in history]
        
        # Plotting
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Losses
        ax1.plot(epochs, total_loss, label='Total Loss', color='#1f77b4', linewidth=2)
        ax1.plot(epochs, box_loss, label='Box Loss (CIoU)', color='#ff7f0e', linestyle='--')
        ax1.plot(epochs, cls_loss, label='Cls Loss (BCE)', color='#2ca02c', linestyle=':')
        ax1.set_title("Training Losses")
        ax1.set_xlabel("Epoch")
        ax1.set_ylabel("Loss")
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # mAP Accuracy
        ax2.plot(epochs, map50, label='mAP@50', color='#9467bd', linewidth=2)
        ax2.set_title("Validation mAP@50")
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("mAP")
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        plt.tight_layout()
        plt.show()
    else:
        print(f"history.json not found in {latest_run}")
else:
    print(f"No run folders found matching name: {EXP_NAME}")

## 4. INT8 Quantization (PTQ & QAT)
Convert your high-precision floating-point (FP32) weights to high-speed integer (INT8) weights for hardware deployment.

### A. Post-Training Quantization (PTQ)
PTQ passes a small set of custom validation images through the pre-trained network to calibrate active node thresholds, then converts the weights to INT8. **It requires no training/backward pass.**

### B. Quantization-Aware Training (QAT)
QAT simulates quantization boundaries during training, allowing the model to adapt and minimize integer rounding errors. This provides the highest mAP accuracy retention under INT8.

In [ ]:
# %% Post-Training Quantization (PTQ) Calibration & ONNX Deployment
# 1. Run PTQ calibration to create the INT8 PyTorch checkpoint (.pt)
best_weights = str(run_folders[-1] / "best.pt") if run_folders else "best.pt"

!python3 scripts/quantize.py \
    --mode ptq \
    --weights {best_weights} \
    --task det \
    --variant {VARIANT} \
    --data {data_yaml} \
    --imgsz {IMGSZ} \
    --n-calib 100 \
    --backend qnnpack

# 2. Export the Float32 model to ONNX using self-healing variant loader
!python3 scripts/export.py \
    --weights {best_weights} \
    --imgsz {IMGSZ} \
    --nc 2 \
    --variant {VARIANT}

# 3. Quantize the ONNX model to INT8 using ONNX Runtime
from pathlib import Path
run_dir = run_folders[-1] if run_folders else Path("experiments/results/crime-detection-yolo-run")
float_onnx = str(run_dir / "exports" / "best.onnx")
int8_onnx = str(run_dir / "quantized" / "model_int8.onnx")

!pip install -q onnxruntime
!python3 scripts/quantize_onnx.py \
    --input {float_onnx} \
    --output {int8_onnx} \
    --mode dynamic


In [ ]:
# %% Quantization-Aware Fine-Tuning (QAT) & ONNX Deployment
# 1. Run QAT training to create the INT8 PyTorch checkpoint (.pt)
best_weights = str(run_folders[-1] / "best.pt") if run_folders else "best.pt"

!python3 scripts/quantize.py \
    --mode qat \
    --weights {best_weights} \
    --task det \
    --variant {VARIANT} \
    --data {data_yaml} \
    --imgsz {IMGSZ} \
    --epochs 5 \
    --lr 1e-4 \
    --backend qnnpack

# 2. Export the Float32 model to ONNX using self-healing variant loader
!python3 scripts/export.py \
    --weights {best_weights} \
    --imgsz {IMGSZ} \
    --nc 2 \
    --variant {VARIANT}

# 3. Quantize the ONNX model to INT8 using ONNX Runtime
from pathlib import Path
run_dir = run_folders[-1] if run_folders else Path("experiments/results/crime-detection-yolo-run")
float_onnx = str(run_dir / "exports" / "best.onnx")
int8_onnx = str(run_dir / "quantized" / "model_int8_qat.onnx")

!pip install -q onnxruntime
!python3 scripts/quantize_onnx.py \
    --input {float_onnx} \
    --output {int8_onnx} \
    --mode dynamic


## 5. Bounding Box Inference Helper
Define the inference function to preprocess custom validation images, feed them to the network, and decode the bboxes with Non-Maximum Suppression (NMS). Supports both FP32 and INT8 models.

In [ ]:
# %% Bounding box inference helper function
import cv2
import torch
from tinyYOLO.models import build_model
from tinyYOLO.utils.postprocess import postprocess_detections

def run_custom_inference(image_path, weights_path, nc, class_names, imgsz=416):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Build model & load state dict
    model, _ = build_model(task='det', variant=VARIANT, nc=nc)
    checkpoint = torch.load(weights_path, map_location=device)
    state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
    
    # Strip '_orig_mod.' (compile), 'module.' (DDP), and 'model.' (quant wrapper) prefixes precisely
    # Filter out thop profiler keys
    cleaned_state_dict = {}
    for k, v in state_dict.items():
        if k.endswith(('total_ops', 'total_params')):
            continue
        k_clean = k
        if k_clean.startswith('_orig_mod.'):
            k_clean = k_clean[len('_orig_mod.'):]
        if k_clean.startswith('module.'):
            k_clean = k_clean[len('module.'):]
        if k_clean.startswith('model.'):
            k_clean = k_clean[len('model.'):]
        cleaned_state_dict[k_clean] = v
        
    model.load_state_dict(cleaned_state_dict)
    model.to(device).eval()
    
    # Preprocess image
    img0 = cv2.imread(str(image_path))
    h0, w0 = img0.shape[:2]
    img_rgb = cv2.cvtColor(img0, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (imgsz, imgsz))
    
    img_tensor = torch.from_numpy(img_resized.transpose(2, 0, 1)).float() / 255.0
    img_tensor = img_tensor.unsqueeze(0).to(device)
    
    with torch.no_grad():
        preds = model(img_tensor)
        
    # Decode predictions & apply NMS
    detections = postprocess_detections(preds, conf_thres=0.25, iou_thres=0.45)[0]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(img_rgb)
    
    if len(detections) > 0:
        print(f"✓ Detected {len(detections)} objects:")
        for det in detections:
            x1, y1, x2, y2 = int(det[0]*w0), int(det[1]*h0), int(det[2]*w0), int(det[3]*h0)
            score = det[4]
            cls_id = int(det[5])
            
            name = class_names[cls_id] if class_names and cls_id in class_names else f"cls_{cls_id}"
            print(f"  - {name}: {score*100:.1f}%")
            
            rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color='#ff0000', linewidth=3)
            ax.add_patch(rect)
            ax.text(x1, y1-10, f"{name} {score:.2f}", color='white', fontsize=12,
                    bbox=dict(facecolor='#ff0000', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2'))
    else:
        print("No objects detected.")
    
    plt.axis('off')
    plt.show()

## 6. Run Bounding Box Inference
Run this cell to fetch the first validation image from your custom dataset, load your trained weights (or your newly generated INT8 quantized weights), run detection, and draw overlays.

In [ ]:
# %% Execute inference on validation sample
# ----------------- SELECT TARGET MODEL WEIGHTS -----------------
# Option 1: Standard FP32 Weights
TARGET_WEIGHTS = str(run_folders[-1] / "best.pt") if run_folders else "best.pt"

# Option 2: INT8 PTQ Quantized Weights (uncomment to test!)
# TARGET_WEIGHTS = str(run_folders[-1] / "quantized" / "model_int8_ptq.pt") if run_folders else ""

# Option 3: INT8 QAT Fine-Tuned Weights (uncomment to test!)
# TARGET_WEIGHTS = str(run_folders[-1] / "quantized" / "model_int8_qat.pt") if run_folders else ""
# ---------------------------------------------------------------

current_nc = nc if 'nc' in locals() else custom_config['nc']
current_names = names if 'names' in locals() else custom_config['names']

if run_folders:
    weights_path = Path(TARGET_WEIGHTS)
    
    # Dynamically find the first image in the validation folder (supporting multiple formats)
    val_images = sorted(list(dataset_dir.glob("val/images/*")))
    if not val_images:
        val_images = sorted(list(dataset_dir.glob("images/val/*")))
    if not val_images:
        val_images = sorted(list(dataset_dir.glob("valid/images/*")))
    if not val_images:
        val_images = sorted(list(dataset_dir.glob("images/valid/*")))

    if weights_path.exists() and val_images:
        print(f"✓ Loading target weights: {weights_path}")
        print(f"✓ Found test image: {val_images[0]}")
        run_custom_inference(
            image_path=val_images[0],
            weights_path=str(weights_path),
            nc=current_nc,
            class_names=current_names,
            imgsz=IMGSZ
        )
    else:
        print(f"⚠️ Error: target weights ({weights_path}) or validation images folder is empty. Please verify paths!")
else:
    print("⚠️ Error: No trained run folders found. Please run the training cell first!")
